In [1]:
%run Latex_macros.ipynb

<IPython.core.display.Latex object>

# Case study: Post-training a model to reason

A *reasoning model* provides responses with a distinctive style
- format
    - *long* Chain of Thought (CoT): step-by-step reasoning
- process
    - *reflection*: looking back at the response so far, and evaluating the solution and strategy
    - *revision*: adapting/changing the current response and strategy

**Reasoning Format example**

**Instruction:**  
"Explain step-by-step how to find the greatest common divisor (GCD) of 48 and 18."

**Expected Reasoning Format:**  
1. **State the problem clearly:** "We want to find the GCD of 48 and 18."  
2. **Describe the method or approach:** "We will use the Euclidean algorithm."  
3. **Stepwise execution:**  
   - Step 1: Divide 48 by 18, the remainder is 12.  
   - Step 2: Divide 18 by 12, the remainder is 6.  
   - Step 3: Divide 12 by 6, the remainder is 0.  
4. **Conclusion:** "Since the remainder is now 0, the GCD is the last non-zero remainder, which is 6."

**Formatted Output:**  
"We want to find the GCD of 48 and 18. Using the Euclidean algorithm,  
Step 1: 48 divided by 18 leaves a remainder of 12.  
Step 2: 18 divided by 12 leaves a remainder of 6.  
Step 3: 12 divided by 6 leaves a remainder of 0.  
Therefore, the GCD of 48 and 18 is 6."



Moreover, for practical reasons
- the reasoning section is bracketed by &lt;think&gt;, &lt;/think&gt; delimiters
- the answer section is bracketed by &lt;answer&gt;, &lt;/answer&gt; delimiters

These makes it possible to hide (at the user option) the reasoning process.

Reasoning behavior is something that is instilled in post-training
- Not the natural behavior of an LLM or Assistant

We will demonstrate how this is done
using Reinforcement Fine Tuning (RFT).


We will use GRPO to impart Reasoning Behavior to a non-Reasoning model.

The following example will be used as a running example to illustrate
the steps of GRPO.

**Prompt**

    Find a number that, when multiplied by 3 and then increased by 7, equals 22.
    
**Base model response**

    The answer is 5 because 15 divided by 3 is 5
    
**Reasoning response**

    <think>
    1. Let the number be x.
    2. The equation is 3x + 7 = 22.
    3. Subtract 7: 3x = 15.
    4. Divide by 3: x = 5.
    </think>
    <answer>5</answer>


In [2]:
import numpy as np

# =================================================================
# 1. INITIALIZATION: PROMPT AND GROUP OUTPUTS
# =================================================================
prompt = "Find a number that, when multiplied by 3 and then increased by 7, equals 22."

# Breakdown of reward components:
# Accuracy: 1.0 (Correct), 0.0 (Incorrect)
# Format: 1.0 (Has <think> and <answer> tags), 0.0 (Missing tags)
# Length Penalty: Negative value based on word count in <think>
outputs = [
    {
        "id": 1,
        "desc": "Correct & Efficient",
        "content": "<think>\n1. Let the number be x.\n2. The equation is 3x + 7 = 22.\n3. Subtract 7: 3x = 15.\n4. Divide by 3: x = 5.\n</think>\n<answer>5</answer>",
        "accuracy": 1.0, "format": 1.0, "len_penalty": -0.10
    },
    {
        "id": 2,
        "desc": "Correct but Rambling",
        "content": "<think>\nI need to find an unknown number. Let's call it 'n'.\nThe problem says if I multiply it by 3, I get 3n.\nThen I increase that by 7, so it becomes 3n + 7.\nThe result must be 22. So 3n + 7 = 22.\nIf I subtract 7 from 22, I get 15.\nNow I have 3n = 15. To find n, I divide 15 by 3.\n15 divided by 3 is 5.\nLet's check: 5 * 3 is 15. 15 + 7 is 22. Correct.\n</think>\n<answer>5</answer>",
        "accuracy": 1.0, "format": 1.0, "len_penalty": -0.50
    },
    {
        "id": 3,
        "desc": "Incorrect Math",
        "content": "<think>\n3x + 7 = 22.\n3x = 22 + 7.\n3x = 29.\nx = 29/3.\n</think>\n<answer>9.66</answer>",
        "accuracy": 0.0, "format": 1.0, "len_penalty": -0.10
    },
    {
        "id": 4,
        "desc": "Concise & Correct",
        "content": "<think>\n(22 - 7) / 3 = 15 / 3 = 5.\n</think>\n<answer>5</answer>",
        "accuracy": 1.0, "format": 1.0, "len_penalty": -0.05
    },
    {
        "id": 5,
        "desc": "Formatting Failure",
        "content": "The answer is 5 because 15 divided by 3 is 5.",
        "accuracy": 1.0, "format": 0.0, "len_penalty": 0.00
    }
]

# Output the prompt and samples for Jupyter display
print(f"PROMPT: {prompt}\n")
print("="*60)
print("GENERATED GROUP SAMPLES")
print("="*60)

for o in outputs:
    print(f"\n[Output {o['id']}: {o['desc']}]")
    print(o['content'])

# =================================================================
# 2. REWARD DECOMPOSITION & ADVANTAGE CALCULATION
# =================================================================
# Calculate total reward for each output
raw_rewards = np.array([o["accuracy"] + o["format"] + o["len_penalty"] for o in outputs])

# Standard GRPO Advantage (z-score) calculation
mean_reward = np.mean(raw_rewards)
std_reward = np.std(raw_rewards)
advantages = (raw_rewards - mean_reward) / (std_reward + 1e-8)

print("\n" + "="*85)
print(f"{'ID':<4} | {'Accuracy':<10} | {'Format':<8} | {'Len Pen':<8} | {'Total':<8} | {'Advantage'}")
print("-" * 85)

for i, o in enumerate(outputs):
    total = raw_rewards[i]
    adv = advantages[i]
    print(f"{o['id']:<4} | {o['accuracy']:<10.1f} | {o['format']:<8.1f} | {o['len_penalty']:<8.2f} | {total:<8.2f} | {adv:+.4f}")

print("-" * 85)
print(f"Group Mean Reward: {mean_reward:.4f}")
print(f"Group Std Dev:     {std_reward:.4f}")


PROMPT: Find a number that, when multiplied by 3 and then increased by 7, equals 22.

GENERATED GROUP SAMPLES

[Output 1: Correct & Efficient]
<think>
1. Let the number be x.
2. The equation is 3x + 7 = 22.
3. Subtract 7: 3x = 15.
4. Divide by 3: x = 5.
</think>
<answer>5</answer>

[Output 2: Correct but Rambling]
<think>
I need to find an unknown number. Let's call it 'n'.
The problem says if I multiply it by 3, I get 3n.
Then I increase that by 7, so it becomes 3n + 7.
The result must be 22. So 3n + 7 = 22.
If I subtract 7 from 22, I get 15.
Now I have 3n = 15. To find n, I divide 15 by 3.
15 divided by 3 is 5.
Let's check: 5 * 3 is 15. 15 + 7 is 22. Correct.
</think>
<answer>5</answer>

[Output 3: Incorrect Math]
<think>
3x + 7 = 22.
3x = 22 + 7.
3x = 29.
x = 29/3.
</think>
<answer>9.66</answer>

[Output 4: Concise & Correct]
<think>
(22 - 7) / 3 = 15 / 3 = 5.
</think>
<answer>5</answer>

[Output 5: Formatting Failure]
The answer is 5 because 15 divided by 3 is 5.

ID   | Accuracy  

## DeepSeek R1 training process

We will motivate the post-training to instill reasoning abilities via the process
used 
- to DeepSeek R1 (reasoning model)
- from DeepSeek-V3 (base model)

We will mention DeepSeek R0 in passing
- as a "failed" attempt at creating a reasoning model using *only* RL and no SFT

The training process has some notable features
- Reward model is rule-based, not trained
    - No need for Human or AI Judge Feedback to define reward behavior
    - Space-efficient: No parameters !
- Remove need to calibrate rewards
    - within questions and across questions
    - replaced with pseudo-ranking within groups
- Synthetic data driven
    - Training data questions/answers obtained from current iteration of model via prompting
- GPRO: new RL training method
    - sample efficient
    

DeepSeek trained R1 to reason using multiple stages
- starting with a large (roughly 700MM parameter) non-reasoning base model: DeepSeek V3

The multiple stages constitute a form of *bootstrapping*
- incremental steps to a final, reasoning model

A preview, in chart form, follows.

Subsequent sections explore each stage in detail.

Here is a picture of the multi-stage process for training DeepSeek R1.

There are 4 stages.  Fortunately: we will discuss each stage separately
- Inputs and Outputs for each stage
- What the stage does
- Why the stage is necessary

<table>
    <img src="images/DeepSeek_r1_training_v1.png" width=100%>
</table>

## How to Interpret the DeepSeek R1 Training Process Flowchart

This flowchart visualizes the DeepSeek R1 training process, using distinct visual cues to represent different components and their relationships. Below is a guide to help you understand the diagram:

### Node Shapes:

*   **Square (`s`)**: Represents a **Stage** in the training process. These are the primary phases of the DeepSeek R1 model's development.
    *   *Examples:* "Stage 1: Initial SFT", "Stage 2: Reasoning-Oriented RL"
*   **Circle (`o`)**: Represents the **Base Model** or foundational component.
    *   *Example:* "Base V3 Model"
*   **Hexagon (`h`)**: Represents **Data Sources** or datasets used in various stages.
    *   *Examples:* "Cold Start Data", "Reasoning-Oriented Data"
*   **Diamond (`d`)**: Represents **Methods** or techniques applied during the training stages.
    *   *Examples:* "SFT", "RL/GRPO", "Rejection Sampling"

### Node Colors:

*   **Pastel Shades**: Each category of nodes (Stages, Base Model, Data Sources, Methods) has a distinct base color to enhance visual differentiation.
    *   **Stages**: Orange/Peach shades (e.g., `#FFDDC1`, `#C1FFD7`, `#C1D7FF`, `#FFC1C1`)
    *   **Base Model**: Light Grey (`#F0F0F0`)
    *   **Data Sources**: Light Blue (`#D4E6F1`)
    *   **Methods**: Light Pink (`#FADBD8`)

### Node Labels:

*   Each node has a clear, descriptive label that may span multiple lines for better readability, indicating the specific stage, data, model, or method it represents.

### Edge Styles and Colors:

Edges (arrows) connect the nodes and represent the flow or relationship between them. Different styles and colors are used to indicate the type of relationship:

*   **Solid Line (`solid`) - Grey**: Indicates a **Sequential Flow** or transition between major training stages.
    *   *Examples:* "Stage1" → "Stage2", "Stage2" → "Stage3"
*   **Dashed Line (`dashed`) - Blue**: Indicates **Data Input**. This shows when a specific dataset is used as input for a training stage.
    *   *Examples:* "Cold Start Data" → "Stage1", "Reasoning-Oriented Data" → "Stage2"
*   **Dotted Line (`dotted`) - Green**: Indicates **Method Application**. This shows when a particular method or technique is applied in a training stage.
    *   *Examples:* "SFT" → "Stage1", "RL/GRPO" → "Stage2"
*   **Dash-Dot Line (`dashdot`) - Black**: Specifically indicates the **Base Model Input**, showing where the foundational model is introduced into the first stage.
    *   *Example:* "Base Model" → "Stage1"

By understanding these visual conventions, you can effectively interpret the intricate training process of the DeepSeek R1 model as depicted in the flowchart.

### DeepSeek R1 Training Process: A High-Level Overview

The DeepSeek R1 training process is a multi-stage approach designed to develop a powerful and aligned language model. The flowchart illustrates this process, highlighting the sequential **Stages**, the foundational **Base Model**, various **Data Sources**, and the **Methods** applied at each step.

Here's a breakdown of the training pipeline:

1.  **Stage 1: Initial SFT (Supervised Fine-Tuning)**
    *   The process begins with the **Base V3 Model** (represented by a **circle**), which serves as the foundation. This model is fine-tuned using **Cold Start Data** (a **hexagon** representing a data source) and the **SFT** (Supervised Fine-Tuning) **Method** (a **diamond**).
    *   *Flowchart References:* The 'Base V3 Model' feeds into 'Stage 1' via a 'dashdot' black edge, and 'Cold Start Data' and 'SFT' feed into 'Stage 1' via 'dashed' blue and 'dotted' green edges, respectively.

2.  **Stage 2: Reasoning-Oriented RL (Reinforcement Learning)**
    *   The model from Stage 1 then progresses to this stage, which focuses on enhancing its reasoning capabilities. It utilizes **Reasoning-Oriented Data** (another **hexagon**) and applies **RL/GRPO** (Reinforcement Learning / General Policy Optimization) and **Rejection Sampling** **Methods** (both **diamonds**).
    *   *Flowchart References:* A 'solid' grey edge connects 'Stage 1' to 'Stage 2'. 'Reasoning-Oriented Data', 'RL/GRPO', and 'Rejection Sampling' are linked to 'Stage 2' by 'dashed' blue and 'dotted' green edges.

3.  **Stage 3: Second SFT (Supervised Fine-Tuning)**
    *   Following the reasoning-oriented RL, the model undergoes a second round of supervised fine-tuning. This stage incorporates **General Purpose Data** (a **hexagon**) and employs the **Distillation Method** (a **diamond**) to refine its overall performance.
    *   *Flowchart References:* 'Stage 2' transitions to 'Stage 3' via a 'solid' grey edge. 'General Purpose Data' and 'Distillation' contribute to 'Stage 3' through 'dashed' blue and 'dotted' green edges.

4.  **Stage 4: Second RL (Reinforcement Learning)**
    *   The final stage aims to align the model with human preferences. It leverages **Preference Data** (a **hexagon**) and applies the **Human Alignment** and **RL/GRPO Methods** (both **diamonds**) to achieve this goal.
    *   *Flowchart References:* 'Stage 3' leads to 'Stage 4' with a 'solid' grey edge. 'Preference Data', 'Human Alignment', and 'RL/GRPO' are inputs to 'Stage 4' via 'dashed' blue and 'dotted' green edges.

Each **Stage** (represented by a **square**) builds upon the previous one, iteratively refining the model's capabilities from initial training to reasoning enhancement and human alignment, guided by specific data inputs and training methods.

# Reinforcement Fine Tuning (RFT): SFT + RL

*Reinforcement Fine Tuning* refers to 
- a post-training method 
- to adapt a model

using Reinforcement Learning **and** Supervised Fine Tuning.


Based on our experience, we might suspect that SFT would be sufficient.
- Our module on Transfer Learning used SFT

But, our motivation for RL suggested that
- Supervised Learning (and SFT) is more about *imitation* of the target of an example
- rather than an understanding of underlying principles

In addition
- there may be *multiple*  answers to a question
    - which are not *syntactically identical* to the target of a training example
    - but which are *acceptable*
    
We are often to rank the degree of acceptability via Preference Data.

For our running example, here are some possible  responses with logically correct answers.

    [Output 1: Correct & Efficient]
    <think>
    1. Let the number be x.
    2. The equation is 3x + 7 = 22.
    3. Subtract 7: 3x = 15.
    4. Divide by 3: x = 5.
    </think>
    <answer>5</answer>

    [Output 2: Correct but Rambling]
    <think>
    I need to find an unknown number. Let's call it 'n'.
    The problem says if I multiply it by 3, I get 3n.
    Then I increase that by 7, so it becomes 3n + 7.
    The result must be 22. So 3n + 7 = 22.
    If I subtract 7 from 22, I get 15.
    Now I have 3n = 15. To find n, I divide 15 by 3.
    15 divided by 3 is 5.
    Let's check: 5 * 3 is 15. 15 + 7 is 22. Correct.
    </think>
    <answer>5</answer>

    [Output 4: Concise & Correct]
    <think>
    (22 - 7) / 3 = 15 / 3 = 5.
    </think>
    <answer>5</answer>

And here is a logically correct answer but that *does not obey* the required format
- bracketing by `<think>` and `<answer>` delimiters


    [Output 5: Formatting Failure]
    The answer is 5 because 15 divided by 3 is 5.
    
Even though it is logically correct
- we prefer the other correct answers that obey formatting

If Supervised Learning is about imitation, Reinforcement Learning is more about teaching principles
- semantic knowledge rather than syntactic imitation

On a more practical level, to instill reasoning behavior
- SFT 
    - requires *many* training examples
    - typically: human labeled
- RL 
    - needs fewer examples
    - iterative improvement with each reward
    - can *re-use* the same example to improve further
        - reward can increase with each re-use
        
So, why doesn't RL suffice for post-training ?

We address this in the next section.

## Stage 1: Supervised Fine Tuning to avoid the "cold start" problem

Components
- Model: Base Model (DeepSeek V3)
- Training data: Cold Start Data (Format examples)
- Method: SFT

Objective
- SFT-trained Model
- Used in Stage 2

<table>
    <img src="images/DeepSeek_r1_training_stage1.png" width=900%>
</table>

**Motivation** (the "Cold Start" problem)

In our running example
- the base model
- has zero probability of producing a correctly formatted answer

The distribution of training examples for the base model
- never encountered the `<think>` and `<answer>` delimiters

This is a major problem for GRPO.

GRPO works by
- sampling a set of $G$ response to a prompt
- pushing the policy toward the responses in the set with high reward

If there is no sample in the set
- with `<think>` and `<answer>` delimiters

we can't adjust the policy in the direction that would produce them.

So GRPO struggles with generating a response in a format for which it was not trained.

They are *out of distribution* relative to  the base model's training data
- Recall: the Fundamental Law of Machine Learning
- so are unlikely to be produced by the base model    

The unmodified base model is unlikely to produce responses in the proper format
- hard-coded rewards based on format are thus zero
- rewards based on correctness less likely for problem instances that require "thinking"

The absence of rewards means
- no learning signal

Sparse rewards compound the problem
- trajectory reward, no intermediate reward
- not much learning per episode
   
Thus, jumping right into  RL may not be productive.

SFT is very good at nudging the base model's outputs to the "new distribution"
- *imitating* the different style of a reasoning response
- even if the responses are not correct

An initial pass of SFT adapts the base model to produce
- a new distribution
- closer to the goal "thinking style" distribution


This overcomes the *Cold Start* problem.

Interestingly, SFT instills
- the *format*
    - step by step
- and *patterns*
    - reflection, revision
- *not necessarily* correctness of reasoning !
    - or at least: correct w.r.t. training examples
    - poor generalization
    
SFT creates a stable base upon which RL can learn to generalize.

### Creating SFT Training examples

The examples needed for SFT Training
- are long CoT responses
- with `<think>` and `<answer>` delimiters

One way to create such examples is via Human involvement
- Recall: RLHF

DeepSeek used a Synthetic Data approach
- prompting an existing model
- to generate responses in the proper format

Two synthetic data approaches were used.
- using an existing model as a generator

*Direct prompting* uses a global (system) prompt to describe the desired output to a question.


A question  $q$ is then presented to the model and its response $a$ is collected
- forming training example $(q,a)$

<table>
<img src="images/DeepSeek_r1_reasoning_prompt.png">
    
<br>
Reference: https://arxiv.org/pdf/2501.12948#page=6
</table>


But the base model might struggle to create the desired format.

As an alternative, *few-shot prompting* was used
- present multiple exemplars of reasoning format question/answer pairs

$$
\begin{array}[lc] \\
(q_1 & , & a_1) & \\
& \vdots \\
(q_k &, & a_k) &\\
\end{array}
$$

**One-shot prompt**

    Below is an example showing how to answer a question with clear structured reasoning including labeled sections.

    For each new question you invent, provide the reasoning answer in the same labeled format.

    **Instruction:**  
    "Explain step-by-step how to find the greatest common divisor (GCD) of 48 and 18."

    **Expected Reasoning Format:**  
    1. **State the problem clearly:** "We want to find the GCD of 48 and 18."  
    2. **Describe the method or approach:** "We will use the Euclidean algorithm."  
    3. **Stepwise execution:**  
       - Step 1: Divide 48 by 18, the remainder is 12.  
       - Step 2: Divide 18 by 12, the remainder is 6.  
       - Step 3: Divide 12 by 6, the remainder is 0.  
    4. **Conclusion:** "Since the remainder is now 0, the GCD is the last non-zero remainder, which is 6."

    **Formatted Output:**  
    "We want to find the GCD of 48 and 18. Using the Euclidean algorithm,  
    Step 1: 48 divided by 18 leaves a remainder of 12.  
    Step 2: 18 divided by 12 leaves a remainder of 6.  
    Step 3: 12 divided by 6 leaves a remainder of 0.  
    Therefore, the GCD of 48 and 18 is 6."

<table>
    <center><strong>Synthetic Generation of Cold Start Data</strong></center>
    <img src="images/DeepSeek_r1_stage_1_data.png">
</table>

| Stage           | Purpose in Reasoning Induction                                  | Training Signal/Data                                   | Strengths                              | Limitations                          |
|:----------------|:---------------------------------------------------------------|:-------------------------------------------------------|:--------------------------------------|:-------------------------------------|
| SFT             | Learn reasoning formats and step-by-step logic                  | Paired (instruction, reasoning chain) examples         | Provides stable, structured output     | Limited generalization, mimicry       |
| RL (e.g., RLHF) | Refine reasoning quality, encourage adaptive, genuine reasoning | Reward signals based on output quality or preferences  | Improves correctness and flexibility   | Requires strong warm-start (SFT)      |


**References for SFT and RL stages**

- [SFT or RL? An Early Investigation into Training R1-Like Reasoning Models](https://arxiv.org/html/2504.11468v1)
- [Dissecting Mathematical Reasoning for LLMs Under Reinforcement Learning](https://arxiv.org/html/2506.04723v1)
- [Beyond Next-Token Prediction: How Post-Training Teaches LLMs to Reason](https://toloka.ai/blog/how-post-training-teaches-llms-to-reason/)


## Stage 2: Reinforcement Learning

Components
- Model: SFT-trained model (Stage 1 output)
- Training data: RL learning examples (thinking traces)
- Method: GRPO

Objective
- Model for Stage 2

<table>
    <img src="images/DeepSeek_r1_training_stage2.png" width=900%>
</table>

The Stage 1 model provides us a more stable base for performing Reinforcement Learning
- to train the model in the *principles* of a correct response
- in addition to a "more correct" format

Why does the format need to be adapted beyond Stage 1 ?

We start Stage 1 with a *non-reasoning* model which typically won't demonstrate
- **long** CoT reasoning
- reasoning behaviors like
    - reflection
    - revision
    
We want to impart better reasoning behavior beyond that of Stage 1.

### Rewards for RL

Reinforcement Learning works via the Environment providing a reward, either in response to
- each action of the Agent (the LLM model)
- or the entire trajectory (response) created by the Agent

In the case of an LLM
- we need a reward for the entire trajectory

Where do these rewards come from ?

Prior approaches often used a "Judge" to determine the quality of a response
- Human: RLHF
- AI: RLAIF

Moreover, multiple responses need to be ranked based on preferences
- to guide the model to the *most preferred* response

Earlier approaches (e.g., PPO with Preference Data)  trained a separate reward model
- based on Human or AI Judges

to imitate the Reward system of the Judge.

DeepSeek took a simpler approach
- *rule-based* rewards

The rule-based reward is usually the sum of several component rewards.
- A reward for proper reasoning format
    - can vary the reward based on min/max number of reasoning steps
- A large *correctness* reward for getting the correct answer
- A penalty (negative reward) to eliminate undesirable artifacts of the model
    - reasoning in a mixture of English/Chinse

For our running example:

We create a Reward for each of the $G$ responses to the prompt.

The *rule-based* reward has several sub-rewards, with different magnitudes.
- Accuracy reward: is the answer correct ?
- Format reward: is the format of the response correct (e.g., delimiters)
    - n.b., without the `<answer>` delimiter: it is not clear how to identify the answer in the response
- Length  penalty: is the response concise/not verbose


    =====================================================================================
    ID   | Accuracy   | Format   | Len Pen  | Total    | Advantage
    -------------------------------------------------------------------------------------
    1    | 1.0        | 1.0      | -0.10    | 1.90     | +1.0270
    2    | 1.0        | 1.0      | -0.50    | 1.50     | +0.1141
    3    | 0.0        | 1.0      | -0.10    | 0.90     | -1.2552
    4    | 1.0        | 1.0      | -0.05    | 1.95     | +1.1411
    5    | 1.0        | 0.0      | 0.00     | 1.00     | -1.0270
    -------------------------------------------------------------------------------------
    Group Mean Reward: 1.4500
    Group Std Dev:     0.4382

The *correctness* reward deserves some discussion.

Some questions have *Verifiable Rewards*
- given question $q$, you may not know *how* to find the correct answer $a$
- but given $a$, you can *verify* that it is correct

For example: finding the square root of an integer, or the prime factors.

For examples without Verifiable Rewards, a Judge can be used for correctness
- Human
- LLM


### Creating RL training examples

Where do the training examples from the RL (Stage 2) come from ?

Synthetic data !

Prompt the Stage 1 model
- similar to the Stage 1 prompt
- but with additional behaviors specified

<table>
    <img src="images/DeepSeek_r1_stage_2_data.png">
</table>

### RL Training

The GRPO method was used
- for each training question $q$
    - use the Synthetic Data approach to create $G$ responses
- GRPO will create a *pseudo-rank* (based on rewards) for each of the $G$ responses
- GRPO will (via Gradient Ascent)
    - increase the probability of generating a preferred (based on rank) response
    - decrease the probability of generating a non-preferred (based on rank) response

For our running example:

the absolute reward for each of the $G$ responses
- is standardized
- resulting in the Advantage


    =====================================================================================
    ID   | Accuracy   | Format   | Len Pen  | Total    | Advantage
    -------------------------------------------------------------------------------------
    1    | 1.0        | 1.0      | -0.10    | 1.90     | +1.0270
    2    | 1.0        | 1.0      | -0.50    | 1.50     | +0.1141
    3    | 0.0        | 1.0      | -0.10    | 0.90     | -1.2552
    4    | 1.0        | 1.0      | -0.05    | 1.95     | +1.1411
    5    | 1.0        | 0.0      | 0.00     | 1.00     | -1.0270
    -------------------------------------------------------------------------------------
    Group Mean Reward: 1.4500
    Group Std Dev:     0.4382
    

The Advantage pushes the policy
- toward *better* response
- not just *good* responses

In the above
- rewards for all responses are positive ("not terrible")
- response 5 has positive reward
    - is "not terrible" (correct format)
    - but not Accurate
- response 5 is Accurate ("good") but *below average*

Without centering the rewards
- the policy would be boosted for *all responses*
- centering results in
    - dis-favoring NOT Accurate response 3
    - favoring Accurate responses 1, 2, and 4
    - dis-favoring Accurate response 5

## Stage 3: Boot-strapping Additional SFT

The Stage 2 model is now able to produce
- long CoT
- with reasoning behaviors
    - reflection and revision

but the answers may not always be correct.

None the less, the model becomes
- an abundant source of training examples
- for an additional round of SFT



### Creating additional SFT Training examples

The Stage 2 model creates SFT examples
- similar to the way the Base Model created Stage 1 SFT training examples

**but** 
- using a much stronger (in reasoning) model: Stage 2 Reasoning Model vs non-reasoning Base Model.

Human filtering/curation of the synthetic examples generated is used to
- filter examples with incorrect responses or undesirable properties
    - *Rejection Sampling*
- to curate a variety of questions

At prior stages only questions
- with reasoning responses and with Verifiable Rewards

were used.

This may have *over-tuned* the model to reasoning.

In Stage 3
- we mix in non-reasoning questions

to ensure the fine-tuned model is not over-specialized.

<table>
    <center><strong>Synthetic Generation of Additional SFT training Data</strong></center>
    <img src="images/DeepSeek_r1_stage_3_data.png">
</table>

## Stage 4: RL for non-reasoning behaviors

Beyond reasoning, there are many other behaviors that can be fine-tuned 
- Helpful
    - behave as an Assistant
    - answer questions and follow instructions
        - versus producing the next token
- Honest
    - fact-based responses
- Harmless
    - respond in a polite, socially-acceptable manner

In Stage 4
- the Stage 3 Reasoning model

is fine-tuned with these additional behaviors.

<table>
    <img src="images/DeepSeek_r1_stage_4_data.png">
</table>

## Distillation of Smaller models

The Base Model used (DeepSeek V3) is a large model.

The advantage of Fine Tuning a large model
- it is a powerful base for Synthetic Data generation

But, in practice, smaller reasoning models are desirable
- for example: to run on local user devices (smart phones) rather than giant,remote compute clusters

The process of Distillation is used
- a large model is prompted with many inputs $q$ to generate a corresponding $a$.
- the large collection of $(q,a)$ pairs generated
    - is an *empirical sample* of the function computed by the large model
- a smaller model is trained on the collection
    - in order to approximate the same function as that computed by the large modelm

In our case
- the large Stage 3 model
- is distilled

into a variety of smaller reasoning models

# Comparison: SFT, RL, RFT


SFT and RL are different methods for fine tuning an LLM.
- RFT combines an initial SFT with a subsequent RL



The distinction becomes every blurrier
- when RL has intermediate rewards
- rather than a single trajectory reward

It is sometimes possible to cast a task into a form appropriate for either SFT or RL with per-step rewards
- Next token prediction
    - SFT: Cross Entropy Loss for every step
    - RL: Per-step reward
        - +1 reward for correct prediction/0 for incorrect prediction

But the choice of which method to use is often dependent on
- the task
- the available training data

It is hard to be precise, but here are some thematic comparisons.

SFT 
- encourages imitation of  the label of an example
    - exact match
- enforces formatting/structure of response
- "surface" level correctness
- well-suited to precisely-defined tasks
    - with *objective* measures of success
    - quantitative measure

RL 
- allows multiple "correct" answers
    - which may be ranked
- "deeper" understanding/generalization
- well-suited to more loosely-defined tasks
    - with *subjective* measures of success
    - *qualitative* measures
    

In terms of training data
- SFT imitation requires *lots* of training examples
    - exploration of alternatives doesn't come into play
- RL can often be accomplished in a very small number of training examples
    - exploration encourage
    
SFT and RL are *complementary* methods for fine tuning an LLM.

| Criteria                 | SFT                                                        | RFT/RLHF                                             |
|:-------------------------|:-----------------------------------------------------------|:-----------------------------------------------------|
| Task type                | Objective, well-defined, clear correct answer tasks         | Subjective, ambiguous, or value-laden tasks          |
| Data availability        | Large, high-quality labeled datasets available              | Little/no labeled data, but feedback/preference signals are available |
| Training complexity      | Simpler (labeled pairs)                                    | More complex (reward model, RL optimization)         |
| Desired outcome          | Accuracy, task performance, factual correctness             | Human preference alignment, style, quality, safety   |
| Overfitting risk         | Higher, if data is limited                                 | Lower; learns general behavior from rewards          |
| Generalization           | Prone to memorization                                      | Promotes adaptability, nuanced behaviors             |
| Cost/resource needs      | Lower; less human-in-the-loop need                         | Higher; human feedback collection and more computation|
| Ideal use cases          | Translation, classification, summarization, retrieval      | Chatbots, open-domain QA, content moderation, dialog |


Here is one rubric:

<table>
<img src="images/rft_vs_sft_decision.png" width=90%>
     
 Reference: https://predibase.com/blog/how-reinforcement-learning-beats-supervised-fine-tuning-when-data-is-scarce
<table>

**References for RFT vs SFT**

- [Why Reinforcement Learning Beats SFT with Limited Data - Predibase](https://predibase.com/blog/how-reinforcement-learning-beats-supervised-fine-tuning-when-data-is-scarce)
- [Preference Alignment vs Supervised Fine-Tuning in LLM Training](https://www.rohan-paul.com/p/preference-alignment-vs-supervised)
- [Supervised Fine-Tuning vs. RLHF: How to Choose the Right Approach](https://www.invisible.co/blog/supervised-fine-tuning-vs-rlhf-how-to-choose-the-right-approach-to-train-your-llm)
- [Fine-Tuning vs RLHF: Choosing the Best LLM Training Method](https://cleverx.com/blog/supervised-fine-tuning-vs-rlhf-choosing-the-right-path-to-train-your-llm)
- [What is supervised fine-tuning? - BlueDot Impact](https://bluedot.org/blog/what-is-supervised-fine-tuning)


# Process Reward Model (PRM) vs Outcome Reward Model (ORM)

Outcome Reward Model (ORM) = Trajectory Reward
- single reward at end of trajectory

Process Reward Model (PRM) = step by step reward

**References for Process Reward Models vs Outcome Reward Models**

- [A Comprehensive Survey of Reward Models: Taxonomy and Applications](https://arxiv.org/html/2504.12328v1)
- [Reward Modeling | RLHF Book by Nathan Lambert](https://rlhfbook.com/c/07-reward-models.html)
- [Let’s Verify Step by Step (OpenAI, Process Supervision)](https://cdn.openai.com/improving-mathematical-reasoning-with-process-supervision/Lets_Verify_Step_by_Step.pdf)
- [Getting LLMs To Reason With Process Rewards](https://patmcguinness.substack.com/p/getting-llms-to-reason-with-process)


# References

| Title (linked)                                                                                                                           | Commentary                                                                                                                                                          |
|:-----------------------------------------------------------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| [InstructGPT: Aligning Language Models with Human Intent via RLHF](https://arxiv.org/abs/2203.02155)                                      | Foundational paper laying out the RLHF approach to align LLMs with human intent using human preference data. Essential for understanding RLHF theory and practice. |
| [A Survey on Post-Training of Large Language Models](https://arxiv.org/abs/2503.06072)                                                    | Comprehensive survey reviewing SFT, RLHF, and newer alignment methods. Synthesizes research trends and challenges in LLM post-training.                          |
| [Reinforcement Learning from AI Feedback (RLAIF): A Scalable Alternative to RLHF](https://arxiv.org/abs/2309.00267)                      | Introduces RLAIF, replacing human feedback with AI-generated feedback for scalable alignment. Critical for understanding automated feedback approaches.           |
| [Constitutional AI: Harmlessness from AI Feedback](https://www.anthropic.com/research/constitutional-ai-harmlessness-from-ai-feedback)   | Proposes Constitutional AI, using a fixed ethical constitution for AI self-critique and revision to improve alignment without human labels.                       |
| [LLM Post-Training: A Deep Dive into Reasoning Large Language Models](https://arxiv.org/abs/2502.21321)                                   | Examines post-training methods focused on improving reasoning in LLMs via SFT and RL, analyzing mechanics and challenges.                                         |
| [SFT Memorizes, RL Generalizes: A Comparative Study of Post-Training Methods for LLMs](https://arxiv.org/abs/2501.17161)                  | Empirically compares SFT and RL in LLMs, showing SFT excels at memorization while RL generalizes better and improves alignment.                                    |
| [How Reinforcement Learning Beats Supervised Fine-Tuning When Data Is Scarce](https://predibase.com/blog/how-reinforcement-learning-beats-supervised-fine-tuning-when-data-is-scarce) | Blog explaining why RL methods can outperform SFT in low-data regimes; offers practical insights for training efficiency.                                        |
| [Beyond Next-Token Prediction: How Post-Training Teaches LLMs to Reason](https://toloka.ai/blog/how-post-training-teaches-llms-to-reason/) | Discusses how combining SFT and RL post-training enables complex reasoning in LLMs, with examples and experimental findings.                                      |
| [Demystifying Reasoning Models](https://cameronrwolfe.substack.com/p/demystifying-reasoning-models)                                       | Blog unpacking the roles of SFT and RL in reasoning capability development; bridges theory and practice with clear explanations.                                  |
| [RLHF vs RLAIF: A Detailed Comparison of AI Training Methods](https://www.sapien.io/blog/rlaif-vs-rlhf-understanding-the-differences)     | Detailed comparison of RLHF and RLAIF approaches, illustrating differences in feedback sources and workflows for AI alignment.                                    |


In [3]:
print("Done")

Done
